# DataTour - Baseline v9

**La Stratégie :**
1. **Le v4** : reprendre exactement la logique v4 qui a fait ses preuve hors-du-temps. On refuse de faire un Targe Encoding sur le destinataire (qui cause le syndrome de la Mule Grillée sur le Leaderboard).
2. **L'injection de la Magie Noire** : Ajouter `orig_error_ratio` et `orig_no_move` (les anomalies comptables) car elles ne souffrent d'aucun surapprentissage temporel. Si un solde ne bouge pas, c'est suspect peu importe l'année.
3. **L'Ensemble Gagnant** : maintenir l'alliance LightGBM + CatBoost de la v4 qui a stabilisé les prédictions.

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from pathlib import Path
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("dataset")
SEED     = 42

train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train : {train.shape}  |  Test : {test.shape}")


Train : (1290081, 11)  |  Test : (430100, 10)


In [3]:
SPLIT_PERIOD = 90

# Aucun tri pour garder l'index pur.
df_tr  = train[train.period <  SPLIT_PERIOD].copy()
df_val = train[train.period >= SPLIT_PERIOD].copy()
print("Split temporel préparé.")


Split temporel préparé.


In [4]:
def build_transactional_features(df):
    df = df.copy()
    eps = 1e-6

    df["is_op03"] = (df["operation"] == "op_03").astype(int)
    df["amount_log"]  = np.log1p(df["amount"])
    df["amount_sqrt"] = np.sqrt(np.maximum(df["amount"], 0))

    df["in_fraud_range"] = ((df["amount"] >= 19000) & (df["amount"] <= 600000)).astype(int)
    df["op03_in_range"]  = (df["is_op03"] & df["in_fraud_range"]).astype(int)

    df["orig_chg"] = df["origin_balance_after"] - df["origin_balance_before"]

    # ---------------- NOUVEAU : ANOMALIES COMPTABLES ROBUSTES ----------------
    df["orig_error_ratio"] = np.abs(df["orig_chg"] + df["amount"]) / (df["amount"] + eps)
    df["has_orig_error"] = (df["orig_error_ratio"] > 0.01).astype(int)
    df["orig_no_move"] = (np.abs(df["orig_chg"]) < 1.0).astype(int)
    # -----------------------------------------------------------------------

    df["amt_to_orig"] = df["amount"] / (df["origin_balance_before"].abs() + eps)
    df["amt_to_dest"] = df["amount"] / (df["destination_balance_before"].abs() + eps)

    return df

def build_account_features(df_target, df_source):
    df = df_target.copy()
    global_mean = df_source["fraud_flag"].mean()
    K = 100 
    eps = 1e-6

    # Seulement Origin Account et Operation (pas Destination !)
    for col in ["origin_account", "operation"]:
        stats = df_source.groupby(col)["fraud_flag"].agg(["sum", "count"])
        smoothed = (stats["sum"] + K * global_mean) / (stats["count"] + K)
        df[f"{col}_te"] = df[col].map(smoothed).fillna(global_mean)

    for col in ["origin_account"]:
        freq = df_source[col].value_counts(normalize=True)
        df[f"{col}_freq"] = df[col].map(freq).fillna(0)

    orig_agg = df_source.groupby("origin_account").agg(
        orig_n_tx=("fraud_flag", "count"),
        orig_n_dest=("destination_account", "nunique"),
        orig_amount_max=("amount", "max"),
    ).fillna(0)

    dest_agg = df_source.groupby("destination_account").agg(
        dest_amount_mean=("amount", "mean"),
        dest_amount_std=("amount", "std"),
    ).fillna(0)

    df = df.join(orig_agg, on="origin_account")
    df = df.join(dest_agg, on="destination_account")

    agg_cols = list(orig_agg.columns) + list(dest_agg.columns)
    for c in agg_cols:
        if c in df.columns:
            df[c] = df[c].fillna(df_source[c].median() if c in df_source.columns else 0)

    # ---------------- NOUVEAU : RATIO COMPORTEMENTAL ROBUSTE ----------------
    df["amt_vs_max"] = df["amount"] / (df["orig_amount_max"] + eps)
    # -----------------------------------------------------------------------

    return df

df_tr  = build_transactional_features(df_tr)
df_val = build_transactional_features(df_val)
test   = build_transactional_features(test)
train  = build_transactional_features(train)

df_val = build_account_features(df_val, df_source=df_tr)
df_tr  = build_account_features(df_tr,  df_source=df_tr)

print(f"Features construites. Train shape: {df_tr.shape}")


Features construites. Train shape: (1138301, 31)


In [5]:
FEATURES = [
    "op03_in_range", "origin_account_te", "operation_te",
    "orig_amount_max", "is_op03", "amount_log", "amount", "amount_sqrt",
    "dest_amount_mean", "orig_n_dest", "amt_to_orig", "amt_to_dest",
    "origin_account_freq", "orig_n_tx", "dest_amount_std", "orig_chg",
    # de nouvelles features injectées
    "orig_error_ratio", "has_orig_error", "orig_no_move", "amt_vs_max"
]

print(f"Nombre de features conservées : {len(FEATURES)}")

X_tr  = df_tr[FEATURES].fillna(0)
y_tr  = df_tr["fraud_flag"]
X_val = df_val[FEATURES].fillna(0)
y_val = df_val["fraud_flag"]


Nombre de features conservées : 20


In [6]:
LGB_PARAMS = dict(
    objective         = "binary",
    metric            = "average_precision",
    learning_rate     = 0.03,
    num_leaves        = 63,
    max_depth         = -1,
    min_child_samples = 100,
    feature_fraction  = 0.7,
    bagging_fraction  = 0.7,
    bagging_freq      = 1,
    lambda_l1         = 1.0,
    lambda_l2         = 1.0,
    min_gain_to_split = 0.01,
    scale_pos_weight  = float((y_tr == 0).sum() / (y_tr == 1).sum()),
    n_estimators      = 3000,
    random_state      = SEED,
    verbose           = -1,
)

model_lgb = lgb.LGBMClassifier(**LGB_PARAMS)
model_lgb.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(200, verbose=False)],
)

val_pred_lgb = model_lgb.predict_proba(X_val)[:, 1]
ap_lgb = average_precision_score(y_val, val_pred_lgb)
print(f"LightGBM Val PR-AUC : {ap_lgb:.5f}")


LightGBM Val PR-AUC : 0.36152


In [7]:
CB_PARAMS = dict(
    iterations       = 3000,
    learning_rate    = 0.03,
    depth            = 6,
    eval_metric      = "PRAUC",
    scale_pos_weight = float((y_tr == 0).sum() / (y_tr == 1).sum()),
    random_seed      = SEED,
    od_type          = "Iter",
    od_wait          = 200,
    verbose          = 0,
)

model_cb = CatBoostClassifier(**CB_PARAMS)
model_cb.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

val_pred_cb = model_cb.predict_proba(X_val)[:, 1]
ap_cb = average_precision_score(y_val, val_pred_cb)
print(f"CatBoost Val PR-AUC : {ap_cb:.5f}")


CatBoost Val PR-AUC : 0.32801


In [8]:
# Validation Ensemble
val_pred_ens = (val_pred_lgb + val_pred_cb) / 2
ap_ens = average_precision_score(y_val, val_pred_ens)
print(f"Ensemble Val PR-AUC : {ap_ens:.5f}")

print("\nEntraînement final (100% du train)...")
train_full = build_account_features(train, df_source=train)
test_full  = build_account_features(test,  df_source=train)
X_full      = train_full[FEATURES].fillna(0)
y_full      = train_full["fraud_flag"]
X_test_full = test_full[FEATURES].fillna(0)

# Entrainement LightGBM (garde le ratio * 1.16 pour Optuna, utiliser best_iteration pur de la v4)
PARAMS_LGB_FINAL = {**LGB_PARAMS}
PARAMS_LGB_FINAL["n_estimators"] = int(model_lgb.best_iteration_ * (105/90))
del PARAMS_LGB_FINAL["metric"]
lgb_final = lgb.LGBMClassifier(**PARAMS_LGB_FINAL)
lgb_final.fit(X_full, y_full)
test_pred_lgb = lgb_final.predict_proba(X_test_full)[:, 1]

# Entrainement CatBoost
PARAMS_CB_FINAL = {**CB_PARAMS}
PARAMS_CB_FINAL["iterations"] = int(max(1, model_cb.get_best_iteration() or 1) * (105/90))
del PARAMS_CB_FINAL["od_type"]
del PARAMS_CB_FINAL["od_wait"]
cb_final = CatBoostClassifier(**PARAMS_CB_FINAL)
cb_final.fit(X_full, y_full)
test_pred_cb = cb_final.predict_proba(X_test_full)[:, 1]

# Moyenne des probabilités
test_pred_ens = np.clip((test_pred_lgb + test_pred_cb) / 2, 0, 1)

submission = pd.DataFrame({"id": test["id"], "target": test_pred_ens})
OUT_FILE = "dataset/submission_v_ultimate.csv"
submission.to_csv(OUT_FILE, index=False)
print(f"{OUT_FILE} généré avec succès !")


Ensemble Val PR-AUC : 0.36142

Entraînement final (100% du train)...
dataset/submission_v_ultimate.csv généré avec succès !
